# AdaptiveSRE Colab Validation Notebook

This notebook is set up for Google Colab validation.

Flow:
1. Install dependencies
2. Clone the repo and start mock services as Python processes
3. Start the main server and verify `/health`
4. Run a short training smoke test
5. Run full training
6. Evaluate the trained checkpoint
7. Generate the reward plot

Success criteria:
- `train.py` completes 10 episodes without crashing
- `positive_trajectories > 0`
- `train.py` completes 200 episodes
- `eval.py` shows Gen1 > Gen0
- `rewards_curve.png` is generated

If Colab fails, the likely culprits are:
- `CUDA out of memory`: reduce `batch_size` in `train.py`
- `unsloth` import error: re-run the install cell
- Mock services do not start: check port conflicts
- `Connection refused` to `localhost:8000`: server did not start
- All rewards are `0.001`: debug `grader.py`
- Model outputs invalid JSON: tune temperature or prompt formatting in `train.py`

In [ ]:
# Cell 1: Install
!pip install -q unsloth trl transformers torch httpx matplotlib
!pip install -q fastapi uvicorn pydantic gradio

In [ ]:
# Cell 2: Clone & start mock services (run them as Python processes since Colab has no Docker)
!git clone https://github.com/ashifsekh/Adaptive-SRE.git
%cd Adaptive-SRE

# Start all 5 mock services in background (since Docker isn't available)
!nohup python3 -m uvicorn mock_services.db.main:app --port 15432 > /dev/null 2>&1 &
!nohup python3 -m uvicorn mock_services.auth.main:app --port 8102 > /dev/null 2>&1 &
!nohup python3 -m uvicorn mock_services.payment.main:app --port 8101 > /dev/null 2>&1 &
!nohup python3 -m uvicorn mock_services.cache.main:app --port 6379 > /dev/null 2>&1 &
!nohup python3 -m uvicorn mock_services.notification.main:app --port 8103 > /dev/null 2>&1 &

# Start main server
!nohup python3 -m uvicorn server.app:app --port 8000 > /dev/null 2>&1 &
!python3 - << 'EOF'
import time
import urllib.request

for _ in range(30):
    try:
        with urllib.request.urlopen('http://localhost:8000/health', timeout=1) as response:
            print(response.read().decode())
        break
    except Exception:
        time.sleep(1)
EOF

In [ ]:
# Cell 3: Short training smoke test (10 episodes, 5-10 mins on T4)
!python train.py --episodes 10 --task easy --output ./test_checkpoints/ --env_url http://localhost:8000

In [ ]:
# Cell 4: If step 3 works, run real training (200 episodes, ~1-2 hours)
!python train.py --episodes 200 --task hard --output ./checkpoints/gen1/ --env_url http://localhost:8000

In [ ]:
# Cell 5: Evaluate
!python eval.py --trained_model ./checkpoints/gen1/final --env_url http://localhost:8000 --output eval_results.json

# Cell 6: Plot
!python plot_rewards.py
from IPython.display import Image
Image('rewards_curve.png')